SPDX-FileCopyrightText: Copyright (c) 1993-2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may not use

this file except in compliance with the License. You may obtain a copy of the License at



http://www.apache.org/licenses/LICENSE-2.0



Unless required by applicable law or agreed to in writing, software

distributed under the License is distributed on an "AS IS" BASIS,

WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

See the License for the specific language governing permissions and

limitations under the License.

# 2. 使用 TensorRT Layer API 构建网络

在本笔记本中，你将学习如何超越预构建的模型格式，直接使用 TensorRT 功能丰富的 Layer API 构建神经网络。这种方法能让你对网络架构和优化进行细粒度的控制。

具体来说，我们将涵盖以下内容：

1.  **从零开始构建循环网络 (LSTM)：** 了解如何定义长短期记忆 (LSTM) 单元的各层，并利用这些组件构建一个完整的循环 LSTM 层。这涉及使用各种 Layer API 功能，如 `add_constant`、`add_matrix_multiply`、`add_elementwise`、`add_slice` 和 `add_activation`。
2.  **实现循环逻辑：** 利用 TensorRT 的 `add_loop` 功能高效处理 LSTM 的循环特性，逐步处理输入序列。
3.  **监控构建进度：** 实现一个 `IProgressMonitor`，实时跟踪引擎创建过程，以便了解可能较长的构建耗时。
4.  **创建版本兼容的引擎：** 学习如何使用 `BuilderFlag.VERSION_COMPATIBLE` 标志保存 TensorRT 引擎，以增强其在不同 TensorRT 补丁版本及兼容硬件上的可移植性。

本示例采用一个小型的单层 LSTM，以便聚焦于这些核心 TensorRT API 特性。我们还将通过与等效的 NumPy 实现进行比对来验证其输出的正确性。

## 简介

虽然通过 ONNX 导入模型十分便捷，但直接使用 TensorRT API 构建网络可实现对网络定义的精细控制。**[TensorRT Layer API](https://docs.nvidia.com/deeplearning/tensorrt/latest/python_api/infer/Graph/Layers.html)** 支持用户显式定义每一层，提供了灵活性与优化空间。

为便于理解与验证，本示例采用小型张量，以便与等效的 NumPy 实现进行直接对比。

> **注意：本示例假定您已熟悉长短期记忆网络（LSTM）的基本概念。若您是 LSTM 新手，建议在继续前先了解其结构与运作原理。**

## Step 0: Prerequisites

In [1]:
# %pip install numpy tensorrt polygraphy --extra-index-url https://pypi.ngc.nvidia.com
import tensorrt as trt
import numpy as np
from typing import Tuple

To simplify, network parameters and weight initializations use small, illustrative values and dimensions.

In [2]:
# === Network Parameters & Weights Initialization ===
batch_size = 1
seq_len = 5      # Length of the sequence
input_size = 1   # Dimension of input vector at each time step
hidden_size = 2  # Dimension of hidden state and cell state
num_units = 1

# --- Create Fixed Dummy Weights and Biases (NumPy arrays with dummy values) ---
# These will be used by both the TensorRT build and the NumPy verification
w_val, u_val, b_val = 0.01, 0.05, 0.3
initial_h_val = 0.1
initial_c_val = 0.2

# Define shapes
w_shape = (input_size, 4 * hidden_size) # e.g., [1, 8] for layer 0
u_shape = (hidden_size, 4 * hidden_size)       # e.g., [2, 8]
b_shape = (4 * hidden_size,)                   # e.g., [8]
initial_h_shape = (batch_size, hidden_size)
initial_c_shape = (batch_size, hidden_size)

# Create NumPy arrays
np_weight_W = np.full(w_shape, w_val, dtype=np.float32)
np_weight_U = np.full(u_shape, u_val, dtype=np.float32)
np_bias = np.full(b_shape, b_val, dtype=np.float32)
np_initial_h = np.full(initial_h_shape, initial_h_val, dtype=np.float32)
np_initial_c = np.full(initial_c_shape, initial_c_val, dtype=np.float32)

# Create inputs for the network
np_inputs = np.ones((seq_len, batch_size, input_size), dtype=np.float32)

print("NumPy Weights Initialized:")
print(f"  W shape : {np_weight_W.shape}")
print(f"  U shape : {np_weight_U.shape}")
print(f"  Bias shape : {np_bias.shape}")
print(f"  Initial H shape : {np_initial_h.shape}")
print(f"  Initial C shape : {np_initial_c.shape}")


NumPy Weights Initialized:
  W shape : (1, 8)
  U shape : (2, 8)
  Bias shape : (8,)
  Initial H shape : (1, 2)
  Initial C shape : (1, 2)


## 步骤 1：使用层API定义LSTM操作

此步骤涉及通过向TensorRT `INetworkDefinition`添加层来定义LSTM操作。

### TensorRT层API的典型使用模式

使用层API向TensorRT网络添加层时，常见模式如下：

1.  **添加层：** 使用 `network.add_*` 方法（例如 `network.add_matrix_multiply`）添加所需层。该方法接收输入张量和层特定参数，并返回一个代表新添加层的 `ILayer` 对象。
2.  **配置层：** 访问返回的 `ILayer` 对象以配置其属性。此为可选步骤，但有助于命名层及其输出张量，从而简化调试并生成更有用的日志。

```python
# 示例：添加并配置一个通用层

# 1. 添加层（替换为具体层，如 add_matrix_multiply）
layer = network.add_some_layer(input_tensor, ...)

# 2. 配置层（可选）
output_tensor = layer.get_output(0)
output_tensor.name = 'my_layer_output'  # 命名输出
# ... 其他配置 ...
```

如需查看可用层类型及其具体方法和属性的完整列表，请参阅官方 [TensorRT Layer API documentation](https://docs.nvidia.com/deeplearning/tensorrt/latest/_static/python-api/infer/Graph/Layers.html).

In [3]:
TRT_LOGGER = trt.Logger(trt.Logger.INFO)

def add_lstm_unit(network: trt.INetworkDefinition,
                  input_x: trt.ITensor,      # Shape: [batch_size, input_size]
                  prev_h: trt.ITensor,       # Shape: [batch_size, hidden_size]
                  prev_c: trt.ITensor,       # Shape: [batch_size, hidden_size]
                  W: np.ndarray,            # Shape: [input_size, 4 * hidden_size]
                  U: np.ndarray,            # Shape: [hidden_size, 4 * hidden_size]
                  bias: np.ndarray,         # Shape: [4 * hidden_size]
                  hidden_size: int,
                  input_size: int
                  ) -> Tuple[trt.ITensor, trt.ITensor]:
    """
    Adds the computations for a single LSTM time step.
    Assumes input tensors have a leading batch dimension.
    """
    batch_size = input_x.shape[0] # Get batch size from input

    # Create constant layers for weights and biases
    W_layer = network.add_constant(W.shape, trt.Weights(W))
    W_layer.get_output(0).name = "W_const"
    U_layer = network.add_constant(U.shape, U)
    U_layer.get_output(0).name = "U_const"
    # Reshape bias for broadcasting: [4*hidden] -> [1, 4*hidden]
    bias_reshaped_np = np.expand_dims(bias.copy(), axis=0)
    bias_layer = network.add_constant(bias_reshaped_np.shape, bias_reshaped_np)
    bias_layer.get_output(0).name = "Bias_const"


    # Linear transformations: Wx = input_x * W ; Uh = prev_h * U
    # Wx = [batch, input] * [input, 4*hidden] = [batch, 4*hidden]
    mm_wx = network.add_matrix_multiply(input_x, trt.MatrixOperation.NONE,
                                        W_layer.get_output(0), trt.MatrixOperation.NONE)
    mm_wx.get_output(0).name = "Wx"

    # Uh = [batch, hidden] * [hidden, 4*hidden] = [batch, 4*hidden]
    mm_uh = network.add_matrix_multiply(prev_h, trt.MatrixOperation.NONE,
                                        U_layer.get_output(0), trt.MatrixOperation.NONE)
    mm_uh.get_output(0).name = "Uh"


    # Combined gates = Wx + Uh + Bias
    gates_wx_uh = network.add_elementwise(mm_wx.get_output(0), mm_uh.get_output(0),
                                         trt.ElementWiseOperation.SUM)
    gates_wx_uh.get_output(0).name = "Wx_plus_Uh"

    gates = network.add_elementwise(gates_wx_uh.get_output(0), bias_layer.get_output(0),
                                    trt.ElementWiseOperation.SUM)

    gates_output = gates.get_output(0) # Shape [batch, 4*hidden]
    gates_output.name = "Gates_Combined"

    # Split the combined gates tensor [batch, 4*hidden] -> four [batch, hidden] gate tensors (Input, Forget, Candidate, Output)
    def add_gate_slice(index):
        gate_slice_layer = network.add_slice(input=gates_output,
                                       start=(0, index * hidden_size), # Start [batch_idx=0, col_idx]
                                       shape=(batch_size, hidden_size), # Slice shape
                                       stride=(1, 1))                   # Stride
        return gate_slice_layer.get_output(0)

    slice_i = add_gate_slice(0)
    slice_i.name = "Slice_I"
    slice_f = add_gate_slice(1)
    slice_f.name = "Slice_F"
    slice_c = add_gate_slice(2)
    slice_c.name = "Slice_C_candidate" # Cell candidate
    slice_o = add_gate_slice(3)
    slice_o.name = "Slice_O"

    # Apply activations
    act_i_layer = network.add_activation(slice_i, trt.ActivationType.SIGMOID)
    act_i = act_i_layer.get_output(0)
    act_i.name = "Gate_I"
    act_f_layer = network.add_activation(slice_f, trt.ActivationType.SIGMOID)
    act_f = act_f_layer.get_output(0)
    act_f.name = "Gate_F"
    act_c_layer = network.add_activation(slice_c, trt.ActivationType.TANH)
    act_c = act_c_layer.get_output(0)
    act_c.name = "Gate_C_candidate"
    act_o_layer = network.add_activation(slice_o, trt.ActivationType.SIGMOID)
    act_o = act_o_layer.get_output(0)
    act_o.name = "Gate_O"

    # Cell state update: c_t = f_t * c_{t-1} + i_t * g_t
    term1_c = network.add_elementwise(act_f, prev_c, trt.ElementWiseOperation.PROD)
    term2_c = network.add_elementwise(act_i, act_c, trt.ElementWiseOperation.PROD)
    next_c_layer = network.add_elementwise(term1_c.get_output(0), term2_c.get_output(0), trt.ElementWiseOperation.SUM)

    next_c = next_c_layer.get_output(0)
    next_c.name = "next_c" # Shape [batch, hidden]

    # Hidden state update: h_t = o_t * tanh(c_t)
    tanh_c_layer = network.add_activation(next_c, trt.ActivationType.TANH)
    tanh_c = tanh_c_layer.get_output(0)
    next_h_layer = network.add_elementwise(act_o, tanh_c, trt.ElementWiseOperation.PROD)

    next_h = next_h_layer.get_output(0)
    next_h.name = "next_h" # Shape [batch, hidden]

    return next_h, next_c


def add_lstm_layer(network: trt.INetworkDefinition,
                   input_sequence: trt.ITensor, # Shape: [seq_len, batch_size, input_size]
                   hidden_size: int,
                   seq_len: int,
                   weight_W: np.ndarray, # [input_size, 4*hidden] or [hidden, 4*hidden]
                   weight_U: np.ndarray, # [hidden, 4*hidden]
                   bias: np.ndarray    # [4*hidden]
                   ) -> trt.ITensor:
    """
    Adds a LSTM to the network by adding one lstm_unit, and run multiple times with loops.
    """
    # Infer batch_size and input_size from the input tensor shape
    assert len(input_sequence.shape) == 3, f"Input sequence tensor must have 3 dimensions [seq, batch, input]. Got shape {input_sequence.shape}"
    input_size = input_sequence.shape[2]

    # Shape: [batch_size, hidden_size]
    initial_h = network.add_constant(np_initial_h.shape, np_initial_h).get_output(0)
    initial_h.name = "Initial_H"
    initial_c = network.add_constant(np_initial_c.shape, np_initial_c).get_output(0)
    initial_c.name = "Initial_C"

    loop = network.add_loop()
    loop.name = "Time_Loop_Layer"

    # add_trip_limit determines when the loop should stop. For here we want the loop to run seq_len times.
    trip_limit = network.add_constant((), np.array([seq_len], dtype=np.int32)).get_output(0)
    loop.add_trip_limit(trip_limit, trt.TripLimit.COUNT)

    # Recurrences for hidden and cell states
    h_recurrence = loop.add_recurrence(initial_h)
    c_recurrence = loop.add_recurrence(initial_c)
    prev_h_tensor = h_recurrence.get_output(0)
    prev_h_tensor.name = "Prev_H"
    prev_c_tensor = c_recurrence.get_output(0)
    prev_c_tensor.name = "Prev_C"

    # add_iterator iterates through slices of the input sequence along the specified axis, providing one slice per iteration.
    x_t_iterator = loop.add_iterator(input_sequence, axis=0)
    x_t = x_t_iterator.get_output(0)
    x_t.name = "x_t"


    # Call the LSTM unit function
    next_h, next_c = add_lstm_unit(network=network,
                                    input_x=x_t,
                                    prev_h=prev_h_tensor,
                                    prev_c=prev_c_tensor,
                                    W=weight_W,
                                    U=weight_U,
                                    bias=bias,
                                    hidden_size=hidden_size,
                                    input_size=input_size)

    # Feed the computed states back into the recurrence inputs
    h_recurrence.set_input(1, next_h)
    c_recurrence.set_input(1, next_c)

    # add_loop_output() collects the values in the loop and outputs them. For this example, we concatenate the values along the first axis.
    loop_output_h = loop.add_loop_output(next_h, trt.LoopOutput.CONCATENATE, axis=0)

    # when using CONCATENATE, the second input must be the trip limit.
    loop_output_h.set_input(1, trip_limit)
    loop_output_h.get_output(0).name = "Hidden_Sequence"

    # --- End of time step loop definition ---

    layer_output_sequence = loop_output_h.get_output(0)

    # The final output sequence is the sequence from the last layer
    if layer_output_sequence is None:
         raise RuntimeError("LSTM layer output was not generated (num_layers may be 0)")
    layer_output_sequence.name = "Final_LSTM_Output_Sequence"
    return layer_output_sequence

## 步骤 2：构建网络

现在我们有了 LSTM 层的实现（`add_lstm_layer`），接下来继续构建 TensorRT `INetworkDefinition`。
这包括通过以下步骤定义网络结构：
1. 使用 `network.add_input` 添加输入张量。
2. 使用自定义的 `add_lstm_layer` 函数添加 LSTM 层。
3. 将 LSTM 层的输出张量标记为网络的最终输出。

In [4]:
builder = trt.Builder(TRT_LOGGER)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED))

# === Network Definition ===
# Shape: [seq_len, batch_size, input_size] -> e.g., [5, 1, 1]
input_tensor = network.add_input(name='input', dtype=trt.float32, shape=(seq_len, batch_size, input_size))

# --- Add SINGLE LSTM Layer ---
lstm_output = add_lstm_layer(network=network,
                                input_sequence=input_tensor,
                                hidden_size=hidden_size,
                                seq_len=seq_len,
                                weight_W=np_weight_W, 
                                weight_U=np_weight_U, 
                                bias=np_bias) 
# lstm_output shape: [seq_len, batch_size, hidden_size] -> e.g., [5, 1, 2]

# --- Mark Output ---
lstm_output.name = 'hidden_state_sequence'
network.mark_output(lstm_output)

[07/03/2026-16:42:26] [TRT] [I] [MemUsageChange] Init CUDA: CPU +18, GPU +0, now: CPU 48, GPU 2793 (MiB)


## 步骤 3：构建引擎

在定义好网络（`INetworkDefinition`）后，下一步是构建优化的 TensorRT 引擎。此过程需要使用 `trt.Builder` 配合一个 `trt.BuilderConfig` 对象来指定引擎的构建方式。

`IBuilderConfig` 允许您控制构建过程的多个方面，例如：
*   设置内存限制（例如，使用 `set_memory_pool_limit` 设置工作空间大小）。
*   设置构建器标志以控制优化策略和兼容性。

准备好网络和配置后，调用 `builder.build_serialized_network(network, config)` 方法生成序列化引擎，随后可将其保存至文件或直接投入使用。

## （可选）定义进度监视器
构建 TensorRT 引擎有时可能耗时较长，尤其是对于复杂模型。如果构建过程看似漫长，请勿担心！TensorRT 提供了一个实用工具 `IProgressMonitor`。该接口支持逐步跟踪构建流程，便于监控进度，甚至在需要时辅助调试。

### 实现 `IProgressMonitor`

要使用进度监视器，需继承 `trt.IProgressMonitor` 并重写其关键方法：

*   `phase_start(self, phase_name, parent_phase, num_steps)`：当构建过程进入重要阶段时（例如“解析 ONNX 模型”、“构建引擎”），TensorRT 会调用此方法。
    *   `phase_name`：当前启动阶段的名称。
    *   `parent_phase`：父阶段的名称（若为子阶段，可为 `None`）。
    *   `num_steps`：该阶段预期的步骤总数。
*   `step_complete(self, phase_name, step)`：当阶段内的每个增量步骤完成后调用。
    *   `phase_name`：当前阶段的名称。
    *   `step`：刚完成的步骤索引（从0开始）。
    *   *你的实现*通常需更新对应的进度指示器。
    *   **关键点：此方法必须返回 `True` 以允许构建继续。** 返回 `False` 或 `None` 将通知 TensorRT 取消构建。
*   `phase_finish(self, phase_name)`：当某个阶段（及其所有步骤）完成时调用。
    *   `phase_name`：已结束阶段的名称。
    *   *你的实现*通常会终止并移除该阶段的进度指示器。

随后，通过 `IBuilderConfig` 挂载监视器：`config.progress_monitor = MyProgressMonitor()`

In [5]:
class SimpleProgressMonitor(trt.IProgressMonitor):
    def __init__(self):
        trt.IProgressMonitor.__init__(self)
        self._active_phases = 0

    def phase_start(self, phase_name, parent_phase, num_steps):
        print(f"[ProgressMonitor] Phase Start: {phase_name} ({num_steps} steps)")
        self._active_phases += 1

    def phase_finish(self, phase_name):
        print(f"[ProgressMonitor] Phase Finish: {phase_name}")
        self._active_phases -= 1

    def step_complete(self, phase_name, step):
        print(f"[ProgressMonitor] Step Complete: {phase_name}, Step {step}")
        return True

    @property
    def active_phases(self):
        return self._active_phases

## （可选）版本兼容引擎
TensorRT引擎通常针对其构建时所使用的特定GPU和TensorRT版本进行了优化。这能最大化性能，但如果部署环境不同，则可能导致不兼容问题。

`trt.BuilderFlag.VERSION_COMPATIBLE` 标志通过创建更具可移植性的引擎来解决此问题。此类引擎对TensorRT版本或GPU型号（在兼容系列内）的微小差异敏感度较低，代价可能是相较于为精确目标优化的非兼容引擎，性能会有所下降。同时，它也减少了每次TensorRT小版本更新时重新构建引擎的需求。版本兼容性自TensorRT 8.6起受支持；计划必须使用至少8.6或更高版本构建，且运行时环境也必须为8.6或更高版本。

### 使用场景
*   在拥有兼容GPU/TRT版本的多样化硬件集群中进行部署。
*   分发至最终用户系统配置各异的应用程序。
*   通过避免频繁重建以适应小版本更新来简化维护工作。

### 工作原理
启用 `trt.BuilderFlag.VERSION_COMPATIBLE` 会指示TensorRT使用更通用的优化策略。默认情况下，此标志还会导致一个名为“精简运行时”（一种特定版本、经过裁剪的运行时组件）的副本被打包到引擎计划文件中。当你在兼容系统上反序列化此引擎计划时，TensorRT会识别嵌入的精简运行时，加载它，并使用该运行时来反序列化和执行计划的其余部分。

由于此过程涉及直接从引擎计划文件加载并执行代码（即精简运行时），你必须明确声明信任该计划的来源和完整性。这需要在尝试反序列化引擎之前，在你的 `trt.Runtime` 实例上设置 `runtime.engine_host_code_allowed = True`。

> **关于多个版本兼容引擎的注意事项：**
如果部署大量版本兼容引擎，每个计划中嵌入的精简运行时可能导致应用程序整体体积过大。另一种替代方案是从引擎计划中排除运行时（使用 `trt.BuilderFlag.EXCLUDE_LEAN_RUNTIME`）并手动加载它。此方法可显著减少总部署占用的空间。有关详细说明，请参阅NVIDIA TensorRT文档中关于https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/advanced.html#manually-loading-the-runtime的部分。

In [ ]:
ENGINE_FILE_PATH = './lstm_network.plan'

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 28) # 256MB
config.progress_monitor = SimpleProgressMonitor()
config.set_flag(trt.BuilderFlag.VERSION_COMPATIBLE)

print("Building engine...")
serialized_engine = builder.build_serialized_network(network, config)

print("Engine build completed.")
with open(ENGINE_FILE_PATH, 'wb') as f:
    f.write(serialized_engine)
print(f"Engine saved to {ENGINE_FILE_PATH}")

## 推理

构建好 TensorRT 引擎后，下一步通常是运行推理以验证其功能与性能。标准流程包括创建运行时（runtime）和执行上下文（execution context）、管理输入输出的 GPU 显存、在主机与设备间传输数据以及执行引擎等操作。尽管该流程提供了细粒度的控制能力，但也涉及大量样板代码。此标准流程已在示例 1 中进行了详细演示。

在本示例中，我们将借助 **https://github.com/NVIDIA/TensorRT/tree/main/tools/Polygraphy** 简化推理流程。这是 TensorRT 内置的多功能工具包，可自动处理诸多底层细节，例如：
*   上下文创建
*   缓冲区管理
*   数据传输

> **重要提示：** Polygraphy 因其易用性非常适合调试与测试场景，但可能会引入额外开销。
> 若需在部署环境中追求最佳性能，建议参考 `1_run_onnx_with_tensorrt` 示例手动编写推理代码。

更多示例请参阅 https://github.com/NVIDIA/TensorRT/tree/main/tools/Polygraphy/examples。

In [ ]:
from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

def run_inference_with_polygraphy(h_input: np.ndarray) -> np.ndarray:
    input_name = 'input'
    output_name = 'hidden_state_sequence'

    # Prepare the feed dictionary required by Polygraphy
    # Ensure input is contiguous C-style array, which Polygraphy prefers.
    h_input_contiguous = np.ascontiguousarray(h_input)
    feed_dict = {input_name: h_input_contiguous}

    print(f"Loading engine from: {ENGINE_FILE_PATH}")
    outputs = None
    load_engine = EngineFromBytes(BytesFromPath(ENGINE_FILE_PATH))
    with TrtRunner(load_engine) as runner:
        outputs = runner.infer(feed_dict=feed_dict)
        # Polygraphy automatically synchronizes, so no explicit stream sync needed here

    output_sequence = outputs[output_name]
    print(f"Output '{output_name}' shape: {output_sequence.shape}, dtype: {output_sequence.dtype}")
    return output_sequence

output_sequence = run_inference_with_polygraphy(np_inputs) 

if output_sequence is not None:
    print(f"\nInput Sequence (shape {np_inputs.shape}):\n{np_inputs}")
    print(f"\nOutput Hidden State Sequence (shape {output_sequence.shape}):\n{output_sequence}")
else:
    print("Inference failed.")
    

## 验证输出（与 NumPy 中等效操作进行对比）

为确保我们的 TensorRT LSTM 实现正确无误，我们将把其输出与 NumPy 中的参考实现进行比对。这是验证自定义层逻辑的常用做法。

该 NumPy 版本将模拟相同的 LSTM 单元计算，并对时间序列展开循环。

In [ ]:
def sigmoid_np(x):
    x_clipped = np.clip(x, -500, 500)  # avoid overflow
    return 1.0 / (1.0 + np.exp(-x_clipped))


def tanh_np(x):
    x_clipped = np.clip(x, -100, 100)  # avoid overflow
    return np.tanh(x_clipped)


def lstm_step_numpy(x_t, prev_h, prev_c, W, U, bias):
    # W: shape [input_size, 4*hidden_size]
    # U: shape [hidden_size, 4*hidden_size]
    # bias: shape [4*hidden_size]
    # x_t: shape [batch_size, input_size]
    # prev_h, prev_c: shape [batch_size, hidden_size]

    hidden_size_ = prev_h.shape[1]

    Wx = x_t @ W  # Shape [batch_size, 4*hidden_size]
    Uh = prev_h @ U  # Shape [batch_size, 4*hidden_size]
    gates = Wx + Uh + bias

    # Split gates
    i = gates[:, 0 * hidden_size_ : 1 * hidden_size_]
    f = gates[:, 1 * hidden_size_ : 2 * hidden_size_]
    c = gates[:, 2 * hidden_size_ : 3 * hidden_size_]  # Cell candidate
    o = gates[:, 3 * hidden_size_ : 4 * hidden_size_]

    i_act = sigmoid_np(i)
    f_act = sigmoid_np(f)
    c_act = tanh_np(c)
    o_act = sigmoid_np(o)

    next_c = f_act * prev_c + i_act * c_act
    next_h = o_act * tanh_np(next_c)

    return next_h, next_c


def lstm_layer_numpy(input_sequence_np, np_W, np_U, np_bias):
    seq_len_ = input_sequence_np.shape[0]
    final_output_sequence_np = None
    h = np_initial_h.copy()
    c = np_initial_c.copy()

    layer_output_sequence_list = []

    for t in range(seq_len_):
        # Slice the input sequence for this time step
        x_t = input_sequence_np[t, :, :]

        h, c = lstm_step_numpy(x_t, h, c, np_W, np_U, np_bias)
        layer_output_sequence_list.append(h)
        layer_output_sequence_np = np.stack(layer_output_sequence_list, axis=0)
        final_output_sequence_np = layer_output_sequence_np

    return final_output_sequence_np


numpy_output_sequence = lstm_layer_numpy(np_inputs, np_weight_W, np_weight_U, np_bias)
print("\n--- NumPy LSTM Calculation Results ---")
print(f"Input Sequence (all ones, shape {np_inputs.shape}):\n{np_inputs}")
print(f"\nNumPy Output Hidden State Sequence (shape {numpy_output_sequence.shape}):\n{numpy_output_sequence}")
print("\n--- Comparison ---")
diff = np.abs(output_sequence - numpy_output_sequence)
max_diff = np.max(diff) if diff.size > 0 else 0.0
print(f"Max absolute difference: {max_diff}")
assert np.allclose(
    output_sequence, numpy_output_sequence, atol=1e-5
), f"Output sequence mismatch between TensorRT and NumPy, max diff: {max_diff}"
print("Notebook executed successfully")

## 总结与后续步骤

恭喜！您已成功完成：
- 使用 TensorRT 的 Layer API 定义了 LSTM 单元与层。
- 通过 `add_loop` 实现了循环逻辑。
- 使用 `IProgressMonitor` 监控引擎构建过程。
- 构建了版本兼容的 TensorRT 引擎。
- 通过 Polygraphy 使用构建的引擎执行推理。
- 对照 NumPy 实现验证了结果。

本示例展示了在 TensorRT 中创建自定义网络架构的基础构建模块。在此基础上，您可以探索：
- 更复杂的网络结构。
- TensorRT API 中提供的各类层。
- 高级循环结构与条件逻辑。
- 若对性能有严格要求，可进一步研究优化技巧（尽管本示例主要侧重 API 用法）。

掌握 Layer API 后，您将能够针对 NVIDIA GPU 推理，优化几乎任意深度学习模型。